# Tanh linearization, residual growth, and condition learning

This standalone notebook tests whether initially small physics residuals—and residuals that rise during learning—are caused by a tanh MLP operating near $\tanh(z)\approx z$ or in its saturated regime.

For a controlled experiment, the network is $x\mapsto u$ and training uses only one physics equation:

$$u_{xx}+u=0.$$

The notebook diagnoses $u$, $u_x$, and $u_{xx}$ separately, but it does **not** optimize them as separate losses. The training objective contains the condition-data loss and the single residual loss $\operatorname{MSE}(u_{xx}+u)$.

Only a tanh MLP is used. We compare the requested input ranges $[-1,1]$ and $[10,20]$ without additional coordinate normalization.


In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

DTYPE = torch.float64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(DTYPE)

# Initialization experiment. Use 30+ seeds for final statistical conclusions.
N_POINTS = 256
N_SEEDS = 10
WIDTH = 32
DEPTH = 3
INIT_SCALES = [0.1, 1.0, 3.0]
DOMAINS = {"[-1, 1]": (-1.0, 1.0), "[10, 20]": (10.0, 20.0)}

# Learning experiment.
TRAIN_SEEDS = list(range(3))
TRAIN_POINTS = 128
TRAIN_STEPS = 5000
LOG_EVERY = 5
LEARNING_RATE = 1e-3
TRAIN_INIT_SCALE = 0.1  # starts with small outputs/residuals
CONDITION_WEIGHT = 1.0
PDE_WEIGHT = 1.0
# Automatically includes the final step when TRAIN_STEPS changes.
SNAPSHOT_STEPS = sorted({0, 10, 50, TRAIN_STEPS // 10, TRAIN_STEPS // 2, TRAIN_STEPS})

root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAVE_DIR = root / "results" / "tanh_linearization"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"device={DEVICE}; output={SAVE_DIR}")


## Model and diagnostics

The network uses tanh in every hidden layer and zero-initialized biases. Besides the residuals, we measure hidden preactivations $z$. `linear_fraction`, `saturated_fraction`, and `relative_tanh_error` test the proposed linearization mechanism directly.


In [ ]:
class MLP(nn.Module):
    def __init__(self, width=32, depth=3, init_scale=1.0):
        super().__init__()
        self.hidden = nn.ModuleList(
            [nn.Linear(1 if i == 0 else width, width) for i in range(depth)]
        )
        self.output = nn.Linear(width, 1)
        for layer in [*self.hidden, self.output]:
            nn.init.xavier_normal_(layer.weight, gain=1.0)
            layer.weight.data.mul_(init_scale)
            nn.init.zeros_(layer.bias)

    def forward(self, x, return_preactivations=False):
        h, zs = x, []
        for layer in self.hidden:
            z = layer(h)
            zs.append(z)
            h = torch.tanh(z)
        u = self.output(h)
        return (u, zs) if return_preactivations else u


def sample_points(bounds, n, seed):
    generator = torch.Generator(device="cpu").manual_seed(seed + 10_000)
    unit = torch.rand((n, 1), generator=generator, dtype=DTYPE)
    lo, hi = bounds
    return (lo + unit * (hi - lo)).to(DEVICE).requires_grad_(True)


def condition_data(bounds):
    """Two boundary observations and one interior anchor from u(x)=sin(x)."""
    lo, hi = bounds
    x = torch.tensor([[lo], [(lo + hi) / 2], [hi]], device=DEVICE, dtype=DTYPE)
    target = torch.sin(x)  # exact solution of u_xx + u = 0
    labels = ["left BC", "interior anchor", "right BC"]
    return x, target, labels

def gradient(outputs, inputs):
    result = torch.autograd.grad(
        outputs, inputs, grad_outputs=torch.ones_like(outputs),
        create_graph=True, retain_graph=True, allow_unused=True,
    )[0]
    return torch.zeros_like(inputs) if result is None else result


def evaluate(model, points):
    u, zs = model(points, True)
    ux = gradient(u, points)
    uxx = gradient(ux, points)
    residual = uxx + u
    losses = {"pde": residual.square().mean()}

    z = torch.cat([item.reshape(-1) for item in zs])
    diagnostics = {
        "u_rms": u.square().mean().sqrt(),
        "ux_rms": ux.square().mean().sqrt(),
        "uxx_rms": uxx.square().mean().sqrt(),
        "residual_rms": residual.square().mean().sqrt(),
        "z_rms": z.square().mean().sqrt(),
        "mean_tanh_slope": (1.0 - torch.tanh(z).square()).mean(),
        "linear_fraction": (z.abs() < 0.5).to(DTYPE).mean(),
        "saturated_fraction": (z.abs() > 2.0).to(DTYPE).mean(),
        "relative_tanh_error": (torch.tanh(z) - z).abs().mean() / (z.abs().mean() + 1e-30),
    }
    return losses, diagnostics

def parameter_gradient_norm(loss, model):
    """Global L2 norm of an unweighted loss term's parameter gradient."""
    gradients = torch.autograd.grad(
        loss, tuple(model.parameters()), retain_graph=True, allow_unused=True,
    )
    squared_norm = sum(
        gradient.detach().square().sum()
        for gradient in gradients if gradient is not None
    )
    return float(torch.sqrt(squared_norm).cpu())


def scalar_dict(mapping):
    return {key: float(value.detach().cpu()) for key, value in mapping.items()}


def quantiles(values):
    return np.quantile(np.asarray(values, dtype=float), [0.25, 0.5, 0.75])


## Experiment A — diagnostics at initialization

Without optimization, measure the RMS magnitudes of $u$, $u_x$, and $u_{xx}$. The final panel shows the actual training loss $\operatorname{MSE}(u_{xx}+u)$, not its RMS, so it is directly comparable with the training plot. This distinguishes a small residual caused by small individual terms from one caused by cancellation between $u_{xx}$ and $u$.


In [ ]:
initialization_results = []
for domain_name, bounds in DOMAINS.items():
    for init_scale in INIT_SCALES:
        for seed in range(N_SEEDS):
            points = sample_points(bounds, N_POINTS, seed)
            torch.manual_seed(seed)
            model = MLP(WIDTH, DEPTH, init_scale).to(DEVICE)
            losses, diagnostics = evaluate(model, points)
            initialization_results.append({
                "domain": domain_name, "init_scale": init_scale, "seed": seed,
                **scalar_dict(losses), **scalar_dict(diagnostics),
            })

path = SAVE_DIR / "initialization_results.csv"
with path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=initialization_results[0].keys())
    writer.writeheader(); writer.writerows(initialization_results)
print(f"wrote {len(initialization_results)} rows to {path}")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(19, 4.5))
xpos = np.arange(len(DOMAINS))
domain_names = list(DOMAINS)
metrics = ["u_rms", "ux_rms", "uxx_rms", "pde"]
titles = [r"RMS($u$)", r"RMS($u_x$)", r"RMS($u_{xx}$)", r"MSE($u_{xx}+u$)"]
for ax, metric, title in zip(axes, metrics, titles):
    for scale, color in zip(INIT_SCALES, ["C0", "C1", "C3"]):
        bands = np.asarray([
            quantiles([r[metric] for r in initialization_results
                       if r["domain"] == domain and r["init_scale"] == scale])
            for domain in domain_names
        ])
        ax.errorbar(xpos, np.maximum(bands[:, 1], 1e-30),
                    yerr=[bands[:, 1]-bands[:, 0], bands[:, 2]-bands[:, 1]],
                    marker="o", color=color, capsize=4, label=f"scale={scale}")
    ax.set_yscale("log"); ax.grid(alpha=.25); ax.set_title(title)
    ax.set_xticks(xpos, domain_names); ax.set_ylabel(title)
axes[0].legend(fontsize=8)
fig.suptitle("Tanh MLP diagnostics at initialization: median and IQR")
fig.tight_layout()
fig.savefig(SAVE_DIR / "initial_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Diagnose whether small derivatives come from linearization or saturation.
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
metrics = ["linear_fraction", "saturated_fraction", "relative_tanh_error", "mean_tanh_slope"]
titles = ["fraction |z|<0.5", "fraction |z|>2",
          "relative tanh linearization error", r"mean tanh slope $1-\tanh^2(z)$"]
for ax, metric, title in zip(axes, metrics, titles):
    for scale, color in zip(INIT_SCALES, ["C0", "C1", "C3"]):
        medians = [np.median([r[metric] for r in initialization_results
                              if r["domain"] == domain and r["init_scale"] == scale])
                   for domain in domain_names]
        ax.plot(xpos, medians, "o-", color=color, label=f"scale={scale}")
    ax.set_title(title); ax.set_xticks(xpos, domain_names); ax.grid(alpha=.25)
axes[0].legend()
fig.tight_layout()
fig.savefig(SAVE_DIR / "activation_diagnostics.png", dpi=180, bbox_inches="tight")
plt.show()


## Experiment B — learn condition points and watch one residual

The condition points are sampled from the exact nontrivial solution $u(x)=\sin(x)$: two endpoint boundary observations and one interior anchor. All three are compatible with $u_{xx}+u=0$.

The fixed objective is

$$L=L_{condition}+L_{PDE},\qquad L_{PDE}=\operatorname{MSE}(u_{xx}+u).$$

Starting with small weights makes the homogeneous residual initially small. Learning the nonzero condition data can make it rise before the network learns the correct curvature and reduces it. The notebook records $u$, $u_x$, and $u_{xx}$ as diagnostics only; they are not separate optimization losses.


At every logged step, the notebook also records the global parameter-gradient norms $||
abla_	heta L_{condition}||_2$ and $||
abla_	heta L_{PDE}||_2$. They are computed before applying the loss weights; the current weights are both one.


In [ ]:
def train_once(domain_name, seed):
    bounds = DOMAINS[domain_name]
    points = sample_points(bounds, TRAIN_POINTS, seed)
    condition_x, condition_target, _ = condition_data(bounds)
    torch.manual_seed(seed)
    model = MLP(WIDTH, DEPTH, TRAIN_INIT_SCALE).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history, snapshots = [], {}

    for step in range(TRAIN_STEPS + 1):
        optimizer.zero_grad(set_to_none=True)
        losses, diagnostics = evaluate(model, points)
        condition_loss = (model(condition_x) - condition_target).square().mean()
        pde_loss = losses["pde"]
        total = CONDITION_WEIGHT * condition_loss + PDE_WEIGHT * pde_loss

        if step % LOG_EVERY == 0 or step == TRAIN_STEPS:
            # These are unweighted global L2 norms over every trainable parameter.
            # retain_graph=True lets the usual total-loss backward pass run below.
            condition_grad_norm = parameter_gradient_norm(condition_loss, model)
            pde_grad_norm = parameter_gradient_norm(pde_loss, model)
            history.append({
                "domain": domain_name, "seed": seed, "step": step,
                "total": float(total.detach().cpu()),
                "condition": float(condition_loss.detach().cpu()),
                "pde": float(pde_loss.detach().cpu()),
                "condition_grad_norm": condition_grad_norm,
                "pde_grad_norm": pde_grad_norm,
                **scalar_dict(diagnostics),
            })
        if seed == TRAIN_SEEDS[0] and step in SNAPSHOT_STEPS:
            grid = torch.linspace(bounds[0], bounds[1], 300, device=DEVICE).reshape(-1, 1)
            with torch.no_grad(): prediction = model(grid).cpu().numpy().ravel()
            snapshots[step] = (grid.cpu().numpy().ravel(), prediction)
        if step == TRAIN_STEPS: break
        total.backward(); optimizer.step()
    return history, snapshots


training_results, model_snapshots = [], {}
for domain_name in DOMAINS:
    for seed in TRAIN_SEEDS:
        history, snapshots = train_once(domain_name, seed)
        training_results.extend(history)
        if snapshots: model_snapshots[domain_name] = snapshots

path = SAVE_DIR / "training_trajectories.csv"
with path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=training_results[0].keys())
    writer.writeheader(); writer.writerows(training_results)
print(f"wrote {len(training_results)} rows to {path}")


In [ ]:
# Losses, component diagnostics, and parameter-gradient norms.
# Curves show the median and IQR across seeds.
fig, axes = plt.subplots(7, 2, figsize=(12, 23), sharex=True)
plot_metrics = [
    "condition", "pde", "u_rms", "ux_rms", "uxx_rms",
    "condition_grad_norm", "pde_grad_norm",
]
ylabels = [
    "condition MSE", r"MSE($u_{xx}+u$)", r"RMS($u$)",
    r"RMS($u_x$)", r"RMS($u_{xx}$)",
    r"$||\nabla_\theta L_{condition}||_2$",
    r"$||\nabla_\theta L_{PDE}||_2$",
]
for row, (metric, ylabel) in enumerate(zip(plot_metrics, ylabels)):
    for col, domain_name in enumerate(DOMAINS):
        ax = axes[row, col]
        selected = [r for r in training_results if r["domain"] == domain_name]
        steps = sorted({r["step"] for r in selected})
        bands = np.asarray([quantiles([r[metric] for r in selected if r["step"] == step])
                            for step in steps])
        ax.plot(steps, np.maximum(bands[:, 1], 1e-30), color="C0", label="tanh")
        ax.fill_between(steps, np.maximum(bands[:, 0], 1e-30),
                        np.maximum(bands[:, 2], 1e-30), color="C0", alpha=.2)
        ax.set_yscale("log"); ax.grid(alpha=.25); ax.set_title(domain_name)
        ax.set_ylabel(ylabel); ax.set_xlabel("optimizer step")
        if row == 0: ax.legend()
fig.suptitle(r"Learning with the single physics loss $u_{xx}+u=0$")
fig.tight_layout()
fig.savefig(SAVE_DIR / "training_losses_diagnostics_and_gradients.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Function snapshots compared with the exact solution.
fig, axes = plt.subplots(1, len(DOMAINS), figsize=(12, 4), squeeze=False)
for col, (domain_name, bounds) in enumerate(DOMAINS.items()):
    ax = axes[0, col]
    snapshots = sorted(model_snapshots[domain_name].items())
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(snapshots)))
    for color, (step, (grid, prediction)) in zip(colors, snapshots):
        ax.plot(grid, prediction, color=color, label=f"step {step:,}")
    exact_x = np.linspace(bounds[0], bounds[1], 500)
    ax.plot(exact_x, np.sin(exact_x), "k--", linewidth=2, label="exact sin(x)")
    cx, cy, labels = condition_data(bounds)
    ax.scatter(cx.cpu(), cy.cpu(), c="black", marker="x", s=60, zorder=5, label="conditions")
    ax.set_title(f"tanh, x∈{domain_name}"); ax.set_xlabel("x"); ax.set_ylabel("u(x)")
    ax.grid(alpha=.25); ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(SAVE_DIR / "model_snapshots.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Quantify whether the PDE residual increases and subsequently decreases.
print("domain | median peak/initial PDE loss | rise-then-fall seeds")
for domain_name in DOMAINS:
    ratios, rise_then_fall = [], 0
    for seed in TRAIN_SEEDS:
        rows = sorted([r for r in training_results if r["domain"] == domain_name
                       and r["seed"] == seed], key=lambda r: r["step"])
        values = np.asarray([r["pde"] for r in rows])
        peak_i = int(np.argmax(values))
        ratios.append(values[peak_i] / max(values[0], 1e-30))
        if peak_i > 0 and peak_i < len(values)-1 and values[-1] < 0.95 * values[peak_i]:
            rise_then_fall += 1
    print(f"{domain_name:8s} | {np.median(ratios):28.3g} | "
          f"{rise_then_fall}/{len(TRAIN_SEEDS)}")


## Interpretation

- Compare `u_rms`, `ux_rms`, and `uxx_rms` with `residual_rms`. A small residual with large $u$ and $u_{xx}$ indicates cancellation; small values of all terms indicate a near-zero network.
- A low mean tanh slope on $[10,20]$ supports saturation as the cause of small derivatives. A high linear fraction and small linearization error support the $\tanh(z)\approx z$ mechanism.
- If PDE loss rises while condition loss falls, data fitting is pulling the model away from its initially near-zero homogeneous solution. A later decrease means the model learned curvature satisfying $u_{xx}\approx-u$.
- The only optimized physics quantity is $\operatorname{MSE}(u_{xx}+u)$. Neither $u$, $u_x$, nor $u_{xx}$ is minimized separately.
- Repeat with 30 or more seeds before drawing conclusions.

- Compare the two parameter-gradient norms, not only the loss values. A term with a small scalar loss can still dominate an update if its parameter gradient is larger. These are global L2 norms and do not show whether the two gradients are aligned or conflicting.
